In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
import warnings
import os
import shap

warnings.filterwarnings('ignore')

# 🔥 모든 그래프의 기본 폰트를 설정하고, 기본 폰트 크기를 18로 선언하여 통일감을 줍니다.
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 18
plt.rcParams['axes.unicode_minus'] = False  

# 1. 저장 디렉토리 안전하게 확인 및 생성
output_dir = r'.\data\figure'
os.makedirs(output_dir, exist_ok=True)

# 2. 데이터 로드 및 전처리
df_filtered = pd.read_csv(r'.\data\c_cleaned_hv_with_features.csv')

# 드롭할 컬럼 정의 및 피처 컬럼 추출
hv_cols = ["HV"]
drop_cols = ["FILE_NAME"] + hv_cols
feat_cols = [c for c in df_filtered.columns if c not in drop_cols and c in df_filtered.columns]

# 결측치 처리 및 독립/종속 변수 분리
df_filtered[feat_cols] = df_filtered[feat_cols].fillna(0)
X = df_filtered[feat_cols].values
y = df_filtered['HV'].values

# 스케일링
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. Gradient Boosting 모델 단독 학습
print("... Gradient Boosting 모델을 학습 중입니다 ...")
gb_model = GradientBoostingRegressor(n_estimators=200, random_state=42)
gb_model.fit(X_scaled, y)

# 4. SHAP Explainer 및 Value 계산
print("... SHAP 값을 계산 중입니다 (TreeExplainer) ...")
explainer = shap.TreeExplainer(gb_model)
shap_values = explainer(X_scaled)

# 시각화 레이블에 원본 피처 이름 주입
shap_values.feature_names = feat_cols


# 5. SHAP 시각화 및 고해상도 저장
print("... SHAP 결과 그래프를 생성 및 저장 중입니다 ...")

# 상위 5개 인덱스 추출
importance_indices = np.argsort(np.abs(shap_values.values).mean(0))[::-1]
top_5_indices = importance_indices[:5]
shap_values_top5 = shap_values[:, top_5_indices]

# -------------------------------------------------------------------------
# [시각화 1] Summary Plot (에러 해결 및 크기 동기화 버전)
# -------------------------------------------------------------------------
fig1 = plt.figure(figsize=(15, 5))

# summary_plot 생성
shap.summary_plot(shap_values_top5, max_display=5, plot_type="dot", show=False)

ax1 = plt.gca()

# 💡 폰트 크기 일치 (눈금 14pt, 레이블 16pt)
ax1.tick_params(axis='both', labelsize=12)
ax1.set_xlabel("SHAP Value (Impact on Predicted HV)", fontsize=12, fontweight='bold', labelpad=12)

# 우측 Colorbar 내부 레이블과 타이틀 크기 일치
if len(fig1.axes) > 1:
    fig1.axes[-1].tick_params(labelsize=12)
    fig1.axes[-1].set_ylabel("Feature Value", fontsize=12, fontweight='bold', labelpad=12)

# 🔥 [핵심 수정] top과 bottom 범위를 타이트하게 좁혀서 피처 간격을 콤팩트하게 압축합니다.
# 기존 top=0.90, bottom=0.18에서 위아래 여백을 대폭 늘려 그래프 본체 영역을 세로로 찌그러트리는 원리입니다.
plt.subplots_adjust(left=0.30, right=0.93, top=0.75, bottom=0.32)

summary_save_path = os.path.join(output_dir, "gb_shap_summary_plot_top5.png")
plt.savefig(summary_save_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"-> Summary Plot 저장 완료: {summary_save_path}")

# -------------------------------------------------------------------------
# [시각화 2] 논문용 그라데이션 Bar Plot
# -------------------------------------------------------------------------
plt.figure(figsize=(15, 5))
ax2 = plt.gca()

top_5_features = [feat_cols[i] for i in top_5_indices]
top_5_values = np.abs(shap_values.values).mean(0)[top_5_indices]

# 시각적 정렬을 위해 순서 역전
top_5_features = top_5_features[::-1]
top_5_values = top_5_values[::-1]

# 그라데이션 컬러 매핑
cmap = plt.cm.get_cmap('viridis')
colors = cmap(top_5_values / max(top_5_values))

# 바 차트 그리기
bars = ax2.barh(top_5_features, top_5_values, color=colors, edgecolor='#222222', height=0.55)

# 💡 축 눈금(18pt) 및 레이블(20pt) 고정
ax2.tick_params(labelsize=18, left=False)
ax2.set_xlabel("Mean |SHAP Value| (Average Impact)", fontsize=20, fontweight='bold', labelpad=12)

ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['left'].set_color('#cccccc')
ax2.spines['bottom'].set_color('#cccccc')

ax2.grid(axis='x', linestyle='--', alpha=0.4, color='#888888')
ax2.set_axisbelow(True)

# 💡 바 우측에 찍히는 수치 텍스트 크기도 18pt로 설정하여 일체감 확보
max_val = max(top_5_values)
for bar in bars:
    width = bar.get_width()
    ax2.text(width + (max_val * 0.012), bar.get_y() + bar.get_height()/2, 
             f'{width:.3f}', 
             va='center', ha='left', fontsize=18, color='#222222', fontweight='bold')

ax2.set_xlim(0, max_val * 1.12)

# 여백 비율 동일하게 지정
plt.subplots_adjust(left=0.30, right=0.93, top=0.90, bottom=0.18)

bar_save_path = os.path.join(output_dir, "gb_shap_bar_plot_top5.png")
plt.savefig(bar_save_path, dpi=300, bbox_inches='tight')
plt.close()
print(f"-> 그라데이션 Bar Plot 저장 완료: {bar_save_path}")

print("\n✅ 두 그래프의 비율, 폰트 종류 및 크기가 완전히 매칭된 고해상도 저장이 완료되었습니다.")

... Gradient Boosting 모델을 학습 중입니다 ...
... SHAP 값을 계산 중입니다 (TreeExplainer) ...
... SHAP 결과 그래프를 생성 및 저장 중입니다 ...
-> Summary Plot 저장 완료: .\data\figure\gb_shap_summary_plot_top5.png
-> 그라데이션 Bar Plot 저장 완료: .\data\figure\gb_shap_bar_plot_top5.png

✅ 두 그래프의 비율, 폰트 종류 및 크기가 완전히 매칭된 고해상도 저장이 완료되었습니다.
